# Picotron: pretrain → infer → SFT → infer

This Kaggle-ready demo trains a deliberately tiny decoder on real FineWeb-Edu text, demonstrates generation, instruction-tunes the same checkpoint on Smol-SmolTalk, and generates again. It is designed for a 2×T4 Kaggle session; this notebook uses one GPU so the whole story remains easy to narrate.

## 1. Clone Picotron

Clone the public repository into Kaggle working storage, then install its declared dependencies.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/AndyMLAndAI/picotron.git picotron
%cd /kaggle/working/picotron
!pip install -q -r requirements.txt

## 2. Extra dependencies and hardware guardrails

`transformers` is used only for the GPT-2 tokenizer. The assertions make the Turing fp16 rule visible: a T4 must select `float16`, never `bfloat16`.

In [ ]:
!pip install -q transformers
import os
import sys
import torch

sys.path.insert(0, '/kaggle/working/picotron/src')
from picotron.utils.hardware import (
    detect_attention_backend,
    detect_triton_support,
    get_gpu_compute_capability,
    select_training_dtype,
)

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running.'
capability = get_gpu_compute_capability(0)
dtype = select_training_dtype(0)
print('GPU:', torch.cuda.get_device_name(0))
print('compute capability:', capability)
print('selected training dtype:', dtype)
print('attention backends:', detect_attention_backend(0))
print('triton (explicit opt-in only):', detect_triton_support(enabled=False, device=0))
assert capability == (7, 5), f'Expected a T4 (7.5), found {capability}.'
assert dtype is torch.float16, 'Turing must select fp16, never bf16.'
assert dtype is not torch.bfloat16
device = torch.device('cuda:0')

## 3. Write the pretraining config

The GPT-2 tokenizer has 50,257 tokens. With untied input/output embeddings, hidden size 10 contributes 1,005,140 embedding parameters; eight tiny blocks add about 13k more, for an approximately 1.02M-parameter model. Layer 3 uses NoPE while the remaining layers use RoPE.

In [ ]:
%%writefile config_pretrain.yaml
# ~1.02M parameters: 2 * 50,257 * 10 untied token/output embeddings
# plus eight small transformer blocks.
checkpoints:
  checkpoint_interval: 3000
  checkpoints_path: /kaggle/working/picotron/checkpoints/pretrain_100m.safetensors
  save_final_state: true
model:
  # auto uses CUDA fp16 autocast on a T4; CPU runs remain fp32.
  dtype: auto
  model_config:
    vocab_size: 50257
    hidden_size: 10
    intermediate_size: 40
    num_hidden_layers: 8
    num_attention_heads: 1
    attention_type: mha
    # Zero-based layer index: layer 3 has NoPE; all other layers retain RoPE.
    nope_layers: [3]
    position_embedding_type: rope
    rope_theta: 10000.0
optimizer:
  learning_rate_scheduler:
    learning_rate: 0.0003
parallelism:
  dp: 1
  zero_stage: 0
tokens:
  sequence_length: 256
  # Four sequences fit safely on one 16 GB T4; 128 OOMs on the logits tensor.
  micro_batch_size: 4
  train_steps: 3000
data:
  dataset_token_path: /kaggle/working/picotron/demo_cache/fineweb_edu_gpt2_100m.uint16
  tokenizer_name: gpt2
  vocab_size: 50257
logging:
  log_level: INFO
  iteration_step_info_interval: 1
general:
  project: picotron
  run: kaggle_pretrain_demo
  seed: 1337

## 4. Stream and cache a bounded FineWeb-Edu slice

The project’s pretraining loader currently consumes ready token tensors, so this notebook uses a bounded streaming-to-cache path: it streams `CC-MAIN-2024-10`, tokenizes until 100M GPT-2 tokens, and writes a compact uint16 token file (~200 MB). This avoids downloading or tokenizing the full corpus up front.

In [ ]:
from pathlib import Path

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

TARGET_TOKENS = 100_000_000
CACHE_DIR = Path('/kaggle/working/picotron/demo_cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TOKEN_PATH = CACHE_DIR / 'fineweb_edu_gpt2_100m.uint16'
tokenizer = AutoTokenizer.from_pretrained('gpt2', use_fast=True)
assert tokenizer.vocab_size == 50257

if not TOKEN_PATH.exists() or TOKEN_PATH.stat().st_size != TARGET_TOKENS * 2:
    written = 0
    stream = load_dataset(
        'HuggingFaceFW/fineweb-edu',
        name='CC-MAIN-2024-10',
        split='train',
        streaming=True,
    )
    with TOKEN_PATH.open('wb') as token_file:
        for example in stream:
            token_ids = tokenizer.encode(example['text'], add_special_tokens=False)
            token_ids.append(tokenizer.eos_token_id)
            remaining = TARGET_TOKENS - written
            if remaining <= 0:
                break
            token_ids = token_ids[:remaining]
            np.asarray(token_ids, dtype=np.uint16).tofile(token_file)
            written += len(token_ids)
            if written and written % 1_000_000 < len(token_ids):
                print(f'cached {written / 1_000_000:.1f}M / 100.0M tokens')
    assert written == TARGET_TOKENS, f'Only cached {written:,} tokens.'

print(f'Prepared {TOKEN_PATH} ({TOKEN_PATH.stat().st_size / 1e6:.1f} MB)')

## 5. Pretrain on real text

This calls Picotron’s core `train()` API, the equivalent of the current CLI path; `scripts/train.py` is intentionally synthetic-data-only. The cache contains 100M tokens, while this T4-safe demo samples 3,000 steps × 4 × 256 = 3.07M training tokens from it. The configured interval writes the final checkpoint on the last step. The current core loop is fp32; hardware detection still reports fp16 correctly for the T4, but AMP wiring is a separate feature.

In [ ]:
from torch.utils.data import DataLoader, Dataset

from picotron.config.config import load_config
from picotron.models.toy_model import ToyDecoderModel
from picotron.training.train_loop import train

class PackedTokenDataset(Dataset):
    def __init__(self, path, sequence_length):
        self.tokens = np.memmap(path, mode='r', dtype=np.uint16)
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.tokens) // self.sequence_length

    def __getitem__(self, index):
        start = index * self.sequence_length
        values = np.asarray(self.tokens[start:start + self.sequence_length], dtype=np.int64)
        return torch.from_numpy(values.copy())

config = load_config('config_pretrain.yaml')
PRETRAIN_STEPS = config.tokens.train_steps
PRETRAIN_CHECKPOINT = config.checkpoints.checkpoints_path
assert PRETRAIN_CHECKPOINT is not None
dataset_token_path = config.data.dataset_token_path
assert dataset_token_path is not None
assert Path(dataset_token_path) == TOKEN_PATH
dataset = PackedTokenDataset(dataset_token_path, config.tokens.sequence_length)
loader = DataLoader(
    dataset, batch_size=config.tokens.micro_batch_size, num_workers=2, pin_memory=True,
)
model = ToyDecoderModel(config)
losses = train(
    model, loader, config, device=device, max_steps=PRETRAIN_STEPS,
    checkpoint_path=PRETRAIN_CHECKPOINT,
)
assert Path(PRETRAIN_CHECKPOINT).exists()
print(f'pretraining loss: {losses[0]:.4f} → {losses[-1]:.4f}')

## 6. Generate after pretraining

This reloads the saved checkpoint into a fresh model and uses greedy decoding. At this scale, output quality will be limited; the important demo signal is that checkpointed pretraining produces a working generation path.

In [ ]:
from torch.optim import AdamW

from picotron.serialize.checkpoint import load_checkpoint

def load_demo_model(checkpoint_path):
    loaded = ToyDecoderModel(config).to(device)
    optimizer = AdamW(
        loaded.parameters(), lr=config.optimizer.learning_rate_scheduler.learning_rate,
    )
    load_checkpoint(loaded, optimizer, checkpoint_path)
    return loaded.eval()

@torch.no_grad()
def generate(model, prompt, max_new_tokens=64):
    token_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    for _ in range(max_new_tokens):
        context = token_ids[:, -config.tokens.sequence_length:]
        next_token = model(context)[:, -1].argmax(dim=-1, keepdim=True)
        token_ids = torch.cat((token_ids, next_token), dim=1)
    return tokenizer.decode(token_ids[0], skip_special_tokens=True)

pretrained_model = load_demo_model(PRETRAIN_CHECKPOINT)
for prompt in ['The capital city of France is', 'Explain why the sky appears blue:']:
    print(f'\nPROMPT: {prompt}\n{generate(pretrained_model, prompt)}')

## 7. Write the SFT demo settings

The SFT stage uses Picotron’s direct `SFTTrainer` API so the notebook can save a final SFT checkpoint for the next generation step. The YAML is intentionally a visible, reproducible record of the run settings.

In [ ]:
%%writefile config_sft.yaml
checkpoints:
  base_checkpoint_path: /kaggle/working/picotron/checkpoints/pretrain_100m.safetensors
  output_checkpoint_path: /kaggle/working/picotron/checkpoints/sft_smoltalk.safetensors
data:
  dataset_name: HuggingFaceTB/smol-smoltalk
  max_examples: 50000
tokens:
  sequence_length: 256
  # Keep SFT within a single T4's memory budget.
  micro_batch_size: 4
  train_steps: 780
optimizer:
  learning_rate_scheduler:
    learning_rate: 0.0001

## 8. Load and format 50k Smol-SmolTalk examples

The maintained dataset identifier is `HuggingFaceTB/smol-smoltalk`. The formatter accepts the usual `messages`/`conversation` schema and fails loudly if the hub schema changes, rather than silently training on the wrong field. Labels equal the conversation tokens; padding labels are ignored.

In [ ]:
import yaml

from datasets import load_dataset

with open('config_sft.yaml', encoding='utf-8') as config_file:
    sft_settings = yaml.safe_load(config_file)
raw_sft = load_dataset(sft_settings['data']['dataset_name'], split='train')
sft_examples = raw_sft.select(
    range(min(sft_settings['data']['max_examples'], len(raw_sft)))
)
print('Smol-SmolTalk columns:', raw_sft.column_names)

def render_conversation(example):
    messages = example.get('messages', example.get('conversation'))
    if not isinstance(messages, list):
        raise KeyError(f'Expected messages/conversation, found {list(example)}')
    rendered = []
    for message in messages:
        role = message.get('role', message.get('from'))
        content = message.get('content', message.get('value'))
        if not isinstance(role, str) or not isinstance(content, str):
            raise ValueError('Each conversation message must contain string role and content.')
        rendered.append(f'<|{role}|>\n{content}')
    return '\n'.join(rendered) + '\n<|assistant|>\n'

class SmolTalkDataset(Dataset):
    def __init__(self, examples, sequence_length):
        self.examples = examples
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        text = render_conversation(self.examples[index])
        ids = tokenizer.encode(text, add_special_tokens=False)[:self.sequence_length]
        return torch.tensor(ids, dtype=torch.long)

def collate_sft(sequences):
    longest = max(sequence.numel() for sequence in sequences)
    input_ids = torch.full((len(sequences), longest), tokenizer.eos_token_id, dtype=torch.long)
    labels = torch.full((len(sequences), longest), -100, dtype=torch.long)
    for row, sequence in enumerate(sequences):
        input_ids[row, :sequence.numel()] = sequence
        labels[row, :sequence.numel()] = sequence
    return {'input_ids': input_ids, 'labels': labels}

sft_dataset = SmolTalkDataset(sft_examples, sft_settings['tokens']['sequence_length'])
sft_loader = DataLoader(
    sft_dataset, batch_size=sft_settings['tokens']['micro_batch_size'], collate_fn=collate_sft,
    num_workers=2, pin_memory=True,
)

## 9. Instruction-tune and save the SFT checkpoint

For SFT we restore only model weights from pretraining and start a fresh AdamW optimizer at the SFT learning rate. That avoids inheriting pretraining optimizer moments or its learning rate. The same Picotron display layer is used here.

In [ ]:
from safetensors.torch import load_file
from picotron.serialize.checkpoint import save_checkpoint
from picotron_sft.sft_trainer import SFTTrainer

sft_model = ToyDecoderModel(config).to(device)
pretrain_weights = Path(sft_settings['checkpoints']['base_checkpoint_path'])
sft_model.load_state_dict(load_file(str(pretrain_weights), device=str(device)))
sft_trainer = SFTTrainer(
    sft_model, sft_loader,
    learning_rate=sft_settings['optimizer']['learning_rate_scheduler']['learning_rate'],
    num_steps=sft_settings['tokens']['train_steps'], display_config=config, device=device,
)
sft_losses = sft_trainer.train()
save_checkpoint(
    sft_trainer.model, sft_trainer.optimizer, len(sft_losses),
    sft_settings['checkpoints']['output_checkpoint_path'],
)
print(f'SFT loss: {sft_losses[0]:.4f} → {sft_losses[-1]:.4f}')
print('saved:', sft_settings['checkpoints']['output_checkpoint_path'])

## 10. Generate after SFT

Use instruction-style prompts that match the formatting used during SFT. With a ~1M-parameter model, expect only a modest qualitative change; describe the comparison honestly in the demo rather than claiming instruction-following quality the model has not earned.

In [ ]:
sft_model = load_demo_model(sft_settings['checkpoints']['output_checkpoint_path'])
for prompt in [
    '<|user|>\nExplain photosynthesis simply.\n<|assistant|>\n',
    '<|user|>\nWrite a short greeting for a new programmer.\n<|assistant|>\n',
]:
    print(f'\nPROMPT: {prompt}\n{generate(sft_model, prompt)}')

print('\nDemo complete.')
print('Pretraining checkpoint:', PRETRAIN_CHECKPOINT)
print('SFT checkpoint:', sft_settings['checkpoints']['output_checkpoint_path'])
print('Narration note: compare structure and instruction-style continuation, not benchmark quality.')